In [1]:
import pandas as pd

df = pd.read_parquet("D:/data/bronze/NFLH/2024/AQUA_MODIS.20240101.L3m.DAY.FLH.nflh.4km.parquet")
print(df.dtypes)
print("----------------------------------------------")
print(df.head())
print("----------------------------------------------")
print(df.shape)          # (列數, 欄數)
print("----------------------------------------------")
print(df.describe())     # 統計摘要


date     object
lat     float32
lon     float32
nflh    float32
dtype: object
----------------------------------------------
         date        lat        lon      nflh
0  2024-01-01  46.145832  37.520840  1.257540
1  2024-01-01  46.145832  37.562508  1.257625
2  2024-01-01  46.145832  37.604172  1.280320
3  2024-01-01  46.145832  37.645840  1.282530
4  2024-01-01  46.145832  37.895840  0.734620
----------------------------------------------
(2569784, 4)
----------------------------------------------
                lat           lon          nflh
count  2.569784e+06  2.569784e+06  2.569784e+06
mean  -9.765082e+00 -1.139378e+01  9.202156e-02
std    2.943321e+01  1.052673e+02  9.186666e-02
min   -7.810417e+01 -1.799792e+02 -3.504753e-05
25%   -3.552084e+01 -9.772916e+01  3.787494e-02
50%   -1.056250e+01 -2.381250e+01  6.702995e-02
75%    1.931250e+01  7.564584e+01  1.170101e-01
max    4.614583e+01  1.799792e+02  2.425270e+00


In [2]:
import xarray as xr
import numpy as np
import pandas as pd

data_par = 'sst'
file_path = 'C:/Users/yah51/Desktop/project/data/CHL/2024/AQUA_MODIS.20241201.L3m.DAY.CHL.chlor_a.4km.nc'

ds = xr.open_dataset(file_path)

ds.info()

xarray.Dataset {
dimensions:
	lat = 4320 ;
	lon = 8640 ;
	rgb = 3 ;
	eightbitcolor = 256 ;

variables:
	float32 chlor_a(lat, lon) ;
		chlor_a:long_name = Chlorophyll Concentration, OCI Algorithm ;
		chlor_a:units = mg m^-3 ;
		chlor_a:standard_name = mass_concentration_of_chlorophyll_in_sea_water ;
		chlor_a:valid_min = 0.0010000000474974513 ;
		chlor_a:valid_max = 100.0 ;
		chlor_a:reference = Hu, C., Lee Z., and Franz, B.A. (2012). Chlorophyll-a algorithms for oligotrophic oceans: A novel approach based on three-band reflectance difference, J. Geophys. Res., 117, C01011, doi:10.1029/2011JC007395. ;
		chlor_a:display_scale = log ;
		chlor_a:display_min = 0.009999999776482582 ;
		chlor_a:display_max = 20.0 ;
	uint8 palette(rgb, eightbitcolor) ;
	float32 lat(lat) ;
		lat:long_name = Latitude ;
		lat:units = degrees_north ;
		lat:standard_name = latitude ;
		lat:valid_min = -90.0 ;
		lat:valid_max = 90.0 ;
	float32 lon(lon) ;
		lon:long_name = Longitude ;
		lon:units = degrees_east ;
		l

In [46]:
# 1. 讀取這個 L3 檔案
ds = xr.open_dataset(file_path)  # 請替換為您的實際路徑

# 2. 提取葉綠素變數的 NumPy 矩陣
data_par_data = ds[data_par].values

# 3. 計算有效值與無效值
total_pixels = data_par_data.size  # 總像素點 (4320 * 8640)
nan_pixels = np.count_nonzero(np.isnan(data_par_data))  # 無效值數量
valid_pixels = total_pixels - nan_pixels  # 有效值數量

# 4. 印出報告
print(f"【全球數據品質報告】")
print(f"總像素點 (Total Pixels)  : {total_pixels:,} 筆")
print(
    f"有效觀測值 (Valid Data)   : {valid_pixels:,} 筆 ({valid_pixels/total_pixels:.2%})"
)
print(
    f"無效值-陸地/雲遮 (NaN Data): {nan_pixels:,} 筆 ({nan_pixels/total_pixels:.2%})"
)

【全球數據品質報告】
總像素點 (Total Pixels)  : 9,331,200 筆
有效觀測值 (Valid Data)   : 1,718,511 筆 (18.42%)
無效值-陸地/雲遮 (NaN Data): 7,612,689 筆 (81.58%)


In [47]:
# 1. 讀取全球 L3 檔案
ds = xr.open_dataset(file_path)

# 2. 擷取日期
file_date = ds.attrs.get("time_coverage_start")[0:10]
file_scale = ds.attrs.get("spatialResolution")[0]

# ==========================================
# 💥 關鍵救星：只抽取 'data_par' 變數，徹底丟棄 26GB 的 palette 垃圾資料！
# ==========================================
ds_chlor_only = ds[[data_par]]

print("安全轉換中，這次絕對不會爆記憶體...")

# 3. 轉成平面表格並剔除 NaN 陸地/雲遮（這樣資料量會再縮減 80%）
df_global = ds_chlor_only.to_dataframe().dropna().reset_index()

# 4. 補上時間戳，並排序成四大黃金欄位
df_global["date"] = file_date
df_global = df_global[["date", "lon", "lat", data_par]]

print("\n--- 恭喜！全球總覽精簡表格轉換成功 (前 5 筆) ---")
print(df_global.head())

# 5. 儲存成全球 CSV 檔
output_filename = f"{file_date}_{file_path.split('.')[5]}_{file_scale}km.csv"
df_global.to_csv(output_filename, index=False)
print(f"檔案已安全儲存至：{output_filename}")

安全轉換中，這次絕對不會爆記憶體...

--- 恭喜！全球總覽精簡表格轉換成功 (前 5 筆) ---
         date        lon        lat   sst
0  2026-05-30 -50.541664  83.791664 -0.21
1  2026-05-30 -50.458328  83.791664 -0.21
2  2026-05-30 -50.374996  83.791664 -0.21
3  2026-05-30 -50.291664  83.791664 -0.21
4  2026-05-30 -50.208328  83.791664 -0.21
檔案已安全儲存至：2026-05-30_sst_9km.csv


In [11]:
# 讀取 CSV 檔案
df = pd.read_csv(output_filename)

# 計算總列數（不包含標頭那一列）
row_count = len(df)

print(f"該 CSV 檔案共有：{row_count:,} 筆有效觀測資料")

該 CSV 檔案共有：1,901,563 筆有效觀測資料


In [ ]:
# 存成 .parquet
# 1. 讀取全球 L3 檔案
ds = xr.open_dataset(file_path)

# 2. 擷取日期
file_date = ds.attrs.get("time_coverage_start")[0:10]

# ==========================================
# 💥 關鍵救星：只抽取 data_par 變數，徹底丟棄 26GB 的 palette 垃圾資料！
# ==========================================
ds_chlor_only = ds[[data_par]]

print("安全轉換中，這次絕對不會爆記憶體...")

# 3. 轉成平面表格並剔除 NaN 陸地/雲遮（這樣資料量會再縮減 80%）
df_global = ds_chlor_only.to_dataframe().dropna().reset_index()

# 4. 補上時間戳，並排序成四大黃金欄位
df_global["date"] = file_date
df_global = df_global[["date", "lon", "lat", data_par]]

print("\n--- 恭喜！全球總覽精簡表格轉換成功 (前 5 筆) ---")
print(df_global.head())

# ==========================================
# 🚀 升級優化：直接改存為 Parquet 格式（不再經過 CSV）
# ==========================================
# 5. 修改副檔名為 .parquet
output_filename = f"{file_date}_{file_path.split('.')[5]}_{file_scale}km.parquet"

# 6. 使用 to_parquet 儲存，採用 snappy 壓縮引擎（速度與壓縮比的最佳平衡）
df_global.to_parquet(output_filename, index=False, compression="snappy")
print(f"原始資料已安全、高效儲存至 Parquet 檔：{output_filename}")

安全轉換中，這次絕對不會爆記憶體...

--- 恭喜！全球總覽精簡表格轉換成功 (前 5 筆) ---
         date        lon        lat   sst
0  2026-05-30 -50.541664  83.791664 -0.21
1  2026-05-30 -50.458328  83.791664 -0.21
2  2026-05-30 -50.374996  83.791664 -0.21
3  2026-05-30 -50.291664  83.791664 -0.21
4  2026-05-30 -50.208328  83.791664 -0.21
原始資料已安全、高效儲存至 Parquet 檔：2026-05-30_sst_9km.parquet


In [ ]:
# 讀取 Parquet 檔案
df = pd.read_parquet(output_filename)

print(df.head())

         date        lon        lat   sst
0  2026-05-30 -50.541664  83.791664 -0.21
1  2026-05-30 -50.458328  83.791664 -0.21
2  2026-05-30 -50.374996  83.791664 -0.21
3  2026-05-30 -50.291664  83.791664 -0.21
4  2026-05-30 -50.208328  83.791664 -0.21
